# PA4 Task 0.3 — Corpus Ingestion into Databricks Vector Search

Run this notebook **inside the Databricks workspace where you have full permissions**
(Unity Catalog write access + Vector Search / AI Search enabled) — not a shared/scoped
instructor workspace. Parses `annual_report.pdf` with `ai_parse_document`, chunks it
with `ai_prep_search`, writes the result to a Delta table, and syncs a Vector Search
Delta Sync index over it.

**Before running:** upload `data/annual_report.pdf` to a Unity Catalog volume (Catalog
Explorer → your catalog → your schema → Volumes → Create/Upload), and fill in the
values in the settings cell below to match your catalog/schema/volume.

In [ ]:
%pip install databricks-vectorsearch python-dotenv
dbutils.library.restartPython()

In [ ]:
import os

# Fill these in to match your workspace — catalog/schema must be ones you can
# CREATE TABLE / CREATE VOLUME in, and the endpoint must already exist (or you
# have permission to create one) under Compute → AI Search.
VOLUME_PATH = "/Volumes/cs4603/default/pa-4/annual_report.pdf"
SOURCE_TABLE = "cs4603.default.emanqureshi_analyst_chunks"
VECTOR_SEARCH_ENDPOINT = "cs4603_rag_endpoint"
VECTOR_SEARCH_INDEX = "cs4603.default.emanqureshi_analyst_index"
EMBEDDINGS_ENDPOINT = "databricks-gte-large-en"

SETTINGS = {
    "vs_endpoint": VECTOR_SEARCH_ENDPOINT,
    "vs_index": VECTOR_SEARCH_INDEX,
    "embeddings": EMBEDDINGS_ENDPOINT,
    "source_table": SOURCE_TABLE,
}

## Function definitions

Mirrors `rag/ingest.py` in the repo (kept in sync manually since this notebook runs
standalone, without the `agent`/`rag` packages installed on the cluster).

In [ ]:
import time


def build_chunks_table(spark, volume_path, chunks_table):
    source_name = volume_path.rsplit("/", 1)[-1]
    spark.sql(f"""
        CREATE OR REPLACE TABLE {chunks_table} AS
        WITH parsed AS (
            SELECT ai_parse_document(content) AS parsed_doc
            FROM read_files('{volume_path}', format => 'binaryFile')
        ),
        chunked AS (
            SELECT explode(
                variant_get(ai_prep_search(parsed_doc), '$.document.contents', 'ARRAY<VARIANT>')
            ) AS chunk
            FROM parsed
        )
        SELECT
            variant_get(chunk, '$.chunk_id', 'STRING') AS chunk_id,
            variant_get(chunk, '$.chunk_to_retrieve', 'STRING') AS chunk_to_retrieve,
            variant_get(chunk, '$.chunk_to_embed', 'STRING') AS chunk_to_embed,
            '{source_name}' AS source,
            variant_get(chunk, '$.pages[0].page_id', 'INT') + 1 AS page
        FROM chunked
    """)
    spark.sql(f"ALTER TABLE {chunks_table} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")


def _wait_for_endpoint_online(client, endpoint_name, timeout_s=600, poll_s=15):
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        state = client.get_endpoint(endpoint_name).get("endpoint_status", {}).get("state")
        if state == "ONLINE":
            return
        time.sleep(poll_s)
    raise TimeoutError(f"Endpoint '{endpoint_name}' did not come ONLINE within {timeout_s}s")


def wait_for_index_ready(index_name, endpoint_name, timeout_s=1800, poll_s=30):
    from databricks.vector_search.client import VectorSearchClient

    client = VectorSearchClient()
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        status = client.get_index(endpoint_name, index_name).describe().get("status", {})
        if status.get("ready"):
            return
        if status.get("detailed_state") == "FAILED":
            raise RuntimeError(f"Index '{index_name}' sync failed: {status}")
        time.sleep(poll_s)
    raise TimeoutError(f"Index '{index_name}' did not become READY within {timeout_s}s")


def create_index(chunks_table):
    from databricks.vector_search.client import VectorSearchClient

    endpoint_name = SETTINGS["vs_endpoint"]
    index_name = SETTINGS["vs_index"]
    client = VectorSearchClient()

    try:
        client.get_endpoint(endpoint_name)
    except Exception:
        client.create_endpoint(name=endpoint_name, endpoint_type="STANDARD")
    _wait_for_endpoint_online(client, endpoint_name)

    try:
        client.get_index(endpoint_name, index_name)
        index_exists = True
    except Exception:
        index_exists = False

    if not index_exists:
        client.create_delta_sync_index(
            endpoint_name=endpoint_name,
            index_name=index_name,
            source_table_name=chunks_table,
            pipeline_type="TRIGGERED",
            primary_key="chunk_id",
            embedding_source_column="chunk_to_retrieve",
            embedding_model_endpoint_name=SETTINGS["embeddings"],
        )
    wait_for_index_ready(index_name, endpoint_name=endpoint_name)


def verify_index(query="net revenue", k=3):
    from databricks.vector_search.client import VectorSearchClient

    client = VectorSearchClient()
    index = client.get_index(SETTINGS["vs_endpoint"], SETTINGS["vs_index"])
    results = index.similarity_search(
        query_text=query,
        columns=["chunk_id", "chunk_to_retrieve", "source", "page"],
        num_results=k,
    )
    return results.get("result", {}).get("data_array", [])

## 1. Parse + chunk the PDF into a Delta table

In [ ]:
build_chunks_table(spark, VOLUME_PATH, SOURCE_TABLE)

In [ ]:
display(
    spark.sql(
        f"SELECT chunk_id, source, page, substring(chunk_to_retrieve, 1, 80) AS preview "
        f"FROM {SOURCE_TABLE} ORDER BY page"
    )
)

## 2. Create (or reuse) the Vector Search endpoint + Delta Sync index

In [ ]:
create_index(SOURCE_TABLE)

## 3. Verify — similarity search smoke test (Task 0.3, step 4)

In [ ]:
results = verify_index("What was Meridian net revenue in FY2023?", k=3)
for r in results:
    print(r)
    print("---")